# Silver Testing and Validation

This notebook validates Silver layer datasets before Gold layer creation.

Validation includes:
- row count checks
- null checks
- invalid value checks
- schema inspection
- sample data inspection

Goal:
Ensure Silver datasets are trusted, consistent, and analytics-ready.

In [0]:
from pyspark.sql.functions import *

catalog = "vattenfall_dev"
refined_schema = "refined"

In [0]:
import yaml
from pyspark.sql import functions as F

config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]
silver_schema = "refined"

silver_tables = [
    "silver_market_prices",
    "silver_weather",
    "silver_grid_events",
    "silver_asset_reference",
    "silver_regional_operations_base"  # optional
]

silver_tables_full = [f"{catalog}.{silver_schema}.{t}" for t in silver_tables]

print("Silver tables to validate:")
for t in silver_tables_full:
    print(t)

In [0]:
existing_tables = []
missing_tables = []

for t in silver_tables_full:
    try:
        df = spark.table(t)  # Try to load table
        count = df.count()   # Row count
        print(f"✅ {t} exists, row count: {count}")
        existing_tables.append(t)
    except Exception as e:
        print(f"❌ Could not access {t}: {e}")
        missing_tables.append(t)

In [0]:
asset_types_allowed = ["TURBINE", "TRANSFORMER", "LINE", "SUBSTATION", "OTHER"]

if f"{catalog}.{silver_schema}.silver_asset_reference" in existing_tables:
    df = spark.table(f"{catalog}.{silver_schema}.silver_asset_reference")
    invalid_df = df.filter(~F.col("asset_type").isin(asset_types_allowed))
    print(f"Invalid asset_type count: {invalid_df.count()}")
    display(invalid_df)

In [0]:
for t in existing_tables:
    print(f"Schema for {t}:")
    spark.table(t).printSchema()

In [0]:
for t in existing_tables:
    print(f"Sample data for {t}:")
    display(spark.table(t).limit(20))

In [0]:
validation_passed = True

# Check missing tables
if missing_tables:
    print("❌ Some Silver tables missing, cannot build Gold:")
    for t in missing_tables:
        print(f"  {t}")
    validation_passed = False

# Check empty tables
for t in existing_tables:
    df = spark.table(t)
    if df.count() == 0:
        print(f"❌ {t} is empty")
        validation_passed = False

# Check invalid asset_type
if f"{catalog}.{silver_schema}.silver_asset_reference" in existing_tables and invalid_df.count() > 0:
    validation_passed = False

if validation_passed:
    print("✅ All Silver tables validated successfully. Gold layer can be built safely.")
else:
    print("❌ Silver validation failed. Fix issues before proceeding to Gold layer.")